In [7]:
import pandas as pd
from pathlib import Path
import re

RAW_DIR = Path('../data/raw')

# Find all assessment files (math/ELA + science)
test_files = sorted([
    f for f in RAW_DIR.glob('*.xlsx')
    if 'assess' in f.name.lower()
])

print(f"Found {len(test_files)} assessment files:\n")
for f in test_files:
    size_mb = f.stat().st_size / 1024 / 1024
    print(f"  [{size_mb:>5.1f} MB]  {f.name}")
    

Found 9 assessment files:

  [ 18.5 MB]  2021_Assessment_updated_redacted (1) (1).xlsx
  [ 66.8 MB]  math_ela_assessments_2022_redacted.xlsx
  [ 63.0 MB]  math_ela_assessments_2023_publish[1].xlsx
  [ 67.6 MB]  math_ela_assessmentsfy24 (2).xlsx
  [ 68.8 MB]  math_ela_assessmentsfy25 (5).xlsx
  [ 15.7 MB]  science_assessments_2022_redacted_0.xlsx
  [ 15.7 MB]  science_assessments_2023_publish[1].xlsx
  [ 16.6 MB]  science_assessments_2024.xlsx
  [ 16.9 MB]  science_assessmentsfy25 (1).xlsx


In [8]:
# Reconnaissance: open each file and report sheet names + dimensions
for f in test_files:
    try:
        xl = pd.ExcelFile(f)
        print(f"\n{'='*80}")
        print(f"FILE: {f.name}")
        print(f"{'='*80}")
        print(f"  Sheets ({len(xl.sheet_names)}): {xl.sheet_names}")
        # Check the first data-looking sheet's dimensions
        for sheet in xl.sheet_names:
            if 'introduction' in sheet.lower() or 'dictionary' in sheet.lower() or 'note' in sheet.lower():
                continue
            df_peek = pd.read_excel(f, sheet_name=sheet, nrows=3)
            print(f"    '{sheet}': {df_peek.shape[1]} cols")
            break
    except Exception as e:
        print(f"\nERROR reading {f.name}: {e}")
        


FILE: 2021_Assessment_updated_redacted (1) (1).xlsx
  Sheets (4): ['School', 'District', 'County', 'State']
    'School': 19 cols

FILE: math_ela_assessments_2022_redacted.xlsx
  Sheets (8): ['Report Introduction', 'Read Me Navigation', 'School', 'District', 'County', 'State', 'Data Dictionary', 'Subgroup Data Dictionary']
    'Read Me Navigation': 0 cols

FILE: math_ela_assessments_2023_publish[1].xlsx
  Sheets (8): ['Report Introduction', 'Read Me Navigation', 'School', 'District', 'County', 'State', 'Data Dictionary', 'Subgroup Data Dictionary']
    'Read Me Navigation': 0 cols

FILE: math_ela_assessmentsfy24 (2).xlsx
  Sheets (7): ['Introduction', 'School', 'District', 'County', 'State', 'Data Dictionary', 'Subgroup Dictionary']
    'School': 20 cols

FILE: math_ela_assessmentsfy25 (5).xlsx
  Sheets (7): ['Introduction', 'School', 'District', 'County', 'State', 'Data Dictionary', 'Subgroup Dictionary']
    'School': 20 cols

FILE: science_assessments_2022_redacted_0.xlsx
  Sheets 

In [9]:
# Look at the District sheet for three sample years to compare schemas
samples = [
    '2021_Assessment_updated_redacted (1) (1).xlsx',
    'math_ela_assessmentsfy24 (2).xlsx',
    'math_ela_assessmentsfy25 (5).xlsx',
]

for fname in samples:
    f = RAW_DIR / fname
    print(f"\n{'='*80}")
    print(f"FILE: {fname}")
    print(f"{'='*80}")
    df = pd.read_excel(f, sheet_name='District', nrows=5)
    print(f"Shape (first 5 rows): {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    print(f"\nFirst 3 rows:")
    with pd.option_context('display.max_columns', None, 'display.width', 200):
        print(df.head(3).to_string())


FILE: 2021_Assessment_updated_redacted (1) (1).xlsx
Shape (first 5 rows): (5, 14)
Columns: ['District Entity ID', 'District CTDS', 'County', 'Subject', 'District Name', 'Subgroup', 'Fiscal Year', 'Test Level', 'Number Tested', 'Percent Passing', 'Percent Proficiency Level 1', 'Percent Proficiency Level 2', 'Percent Proficiency Level 3', 'Percent Proficiency Level 4']

First 3 rows:
   District Entity ID  District CTDS  County                Subject              District Name      Subgroup  Fiscal Year           Test Level Number Tested Percent Passing Percent Proficiency Level 1 Percent Proficiency Level 2 Percent Proficiency Level 3 Percent Proficiency Level 4
0                4153       10201000  Apache  English Language Arts  St Johns Unified District  All Students         2021  All Alt Assessments            12               *                           *                           *                           *                           *
1                4153       10201000  Apache

In [10]:
import re

def extract_year_from_file(filepath):
    """Pull fiscal year from filename — handles ADE's inconsistent naming."""
    name = filepath.name.lower()
    # Try to match patterns like 2021, 2022, fy24, fy25
    for pat in [r'(\d{4})', r'fy(\d{2})']:
        m = re.search(pat, name)
        if m:
            year_str = m.group(1)
            year = int(year_str) if len(year_str) == 4 else 2000 + int(year_str)
            if 2019 <= year <= 2030:
                return year
    return None

def normalize_columns(df):
    """Standardize column names across years."""
    rename_map = {
        'Fiscal Year': 'FiscalYear',
        'District CTDS': 'DistrictCTDS',
        'District Name': 'DistrictName',
        'School Entity ID': 'SchoolEntityID',
        'School Name': 'SchoolName',
        'School CTDS': 'SchoolCTDS',
    }
    df = df.rename(columns={c: rename_map.get(c, c) for c in df.columns})
    df.columns = [c.strip() for c in df.columns]
    return df

# Probe each file's "year" assignment to verify the mystery science_assessments_0 case
print("Year detection per file:")
for f in test_files:
    detected = extract_year_from_file(f)
    print(f"  {f.name:60s} -> {detected}")

Year detection per file:
  2021_Assessment_updated_redacted (1) (1).xlsx                -> 2021
  math_ela_assessments_2022_redacted.xlsx                      -> 2022
  math_ela_assessments_2023_publish[1].xlsx                    -> 2023
  math_ela_assessmentsfy24 (2).xlsx                            -> 2024
  math_ela_assessmentsfy25 (5).xlsx                            -> 2025
  science_assessments_2022_redacted_0.xlsx                     -> 2022
  science_assessments_2023_publish[1].xlsx                     -> 2023
  science_assessments_2024.xlsx                                -> 2024
  science_assessmentsfy25 (1).xlsx                             -> 2025


In [11]:
# Subgroups we care about for analysis
SUBGROUPS_OF_INTEREST = {
    'All Students',
    'White',
    'Black or African American',
    'Hispanic or Latino',
    'American Indian or Alaska Native',
    'Asian',
    'Homeless',
    'Income Eligibility 1 and 2',
    'ELFEP14',
    'Limited English Proficient',
    'Students with Disabilities',
}

# Test levels we care about — the aggregate, plus grade-specific for high school
TEST_LEVELS_OF_INTEREST = {
    'All Assessments',  # the headline aggregate
    'ACT Grade 11',     # high school ELA/Math
    'AzSCI Grade 11',   # high school science
    'AASA Grade 8',     # 8th grade — useful as predictor for follow-up analysis
}

def load_test_data_at_level(filepath, level):
    """Load one file's data at a given level (School/District/State)."""
    df = pd.read_excel(filepath, sheet_name=level)
    df = normalize_columns(df)
    
    # Apply our filters early to keep the dataframe small
    if 'Subgroup' in df.columns:
        df = df[df['Subgroup'].isin(SUBGROUPS_OF_INTEREST)]
    if 'Test Level' in df.columns:
        df = df[df['Test Level'].isin(TEST_LEVELS_OF_INTEREST)]
    # Keep only Full Academic Year if the column exists (2022+)
    if 'FAY Status' in df.columns:
        df = df[df['FAY Status'] == 'Full Academic Year']
    
    df['source_file'] = filepath.name
    df['detected_year'] = extract_year_from_file(filepath)
    return df

# Load all years × all levels
print("Loading test data... (this takes 1–2 minutes)\n")

datasets = {'School': [], 'District': [], 'State': []}

for f in test_files:
    year = extract_year_from_file(f)
    if year is None:
        print(f"  SKIPPED (no year): {f.name}")
        continue
    
    for level in ['School', 'District', 'State']:
        try:
            df = load_test_data_at_level(f, level)
            datasets[level].append(df)
            print(f"  {f.name[:50]:50s} | {level:8s} | {len(df):>7,} rows")
        except Exception as e:
            print(f"  ERROR: {f.name} {level}: {e}")

# Concatenate
schools_test = pd.concat(datasets['School'], ignore_index=True, sort=False) if datasets['School'] else pd.DataFrame()
districts_test = pd.concat(datasets['District'], ignore_index=True, sort=False) if datasets['District'] else pd.DataFrame()
state_test = pd.concat(datasets['State'], ignore_index=True, sort=False) if datasets['State'] else pd.DataFrame()

print(f"\n{'='*60}")
print(f"COMBINED:")
print(f"  Schools:   {len(schools_test):>8,} rows")
print(f"  Districts: {len(districts_test):>8,} rows")
print(f"  State:     {len(state_test):>8,} rows")

print(f"\nDistricts sample (first 3 rows):")
with pd.option_context('display.max_columns', None, 'display.width', 200):
    print(districts_test.head(3).to_string())

Loading test data... (this takes 1–2 minutes)

  2021_Assessment_updated_redacted (1) (1).xlsx      | School   |  22,381 rows
  2021_Assessment_updated_redacted (1) (1).xlsx      | District |   6,961 rows
  2021_Assessment_updated_redacted (1) (1).xlsx      | State    |      56 rows
  math_ela_assessments_2022_redacted.xlsx            | School   |       0 rows
  math_ela_assessments_2022_redacted.xlsx            | District |       0 rows
  math_ela_assessments_2022_redacted.xlsx            | State    |       0 rows
  math_ela_assessments_2023_publish[1].xlsx          | School   |       0 rows
  math_ela_assessments_2023_publish[1].xlsx          | District |       0 rows
  math_ela_assessments_2023_publish[1].xlsx          | State    |       0 rows
  math_ela_assessmentsfy24 (2).xlsx                  | School   |       0 rows
  math_ela_assessmentsfy24 (2).xlsx                  | District |       0 rows
  math_ela_assessmentsfy24 (2).xlsx                  | State    |       0 rows
  mat

In [12]:
# Find out what values FAY Status actually contains in 2022+ files
sample_files = [
    'math_ela_assessmentsfy25 (5).xlsx',
    'math_ela_assessmentsfy24 (2).xlsx',
    'math_ela_assessments_2022_redacted.xlsx',
]

for fname in sample_files:
    f = RAW_DIR / fname
    df = pd.read_excel(f, sheet_name='District', nrows=2000)
    df = normalize_columns(df)
    print(f"\n{fname}:")
    if 'FAY Status' in df.columns:
        print(f"  Unique FAY Status values: {df['FAY Status'].unique().tolist()}")
        print(f"  Value counts:")
        print(df['FAY Status'].value_counts().to_string())
    else:
        print(f"  No 'FAY Status' column. Columns: {df.columns.tolist()}")
        


math_ela_assessmentsfy25 (5).xlsx:
  Unique FAY Status values: ['All']
  Value counts:
FAY Status
All    2000

math_ela_assessmentsfy24 (2).xlsx:
  Unique FAY Status values: ['All']
  Value counts:
FAY Status
All    2000

math_ela_assessments_2022_redacted.xlsx:
  Unique FAY Status values: ['FAY']
  Value counts:
FAY Status
FAY    2000


In [13]:
# Check ALL FAY Status values across the full file (not just first 2000 rows)
for fname in sample_files:
    f = RAW_DIR / fname
    df_full = pd.read_excel(f, sheet_name='District')
    df_full = normalize_columns(df_full)
    print(f"\n{fname}:")
    print(f"  Total rows: {len(df_full):,}")
    if 'FAY Status' in df_full.columns:
        print(f"  Unique FAY Status values: {df_full['FAY Status'].unique().tolist()}")
        print(f"  Value counts:")
        print(df_full['FAY Status'].value_counts().to_string())


math_ela_assessmentsfy25 (5).xlsx:
  Total rows: 278,412
  Unique FAY Status values: ['All', 'FAY', 'Not FAY']
  Value counts:
FAY Status
All        111168
FAY        107947
Not FAY     59297

math_ela_assessmentsfy24 (2).xlsx:
  Total rows: 273,583
  Unique FAY Status values: ['All', 'FAY', 'Not FAY']
  Value counts:
FAY Status
All        109346
FAY        106075
Not FAY     58162

math_ela_assessments_2022_redacted.xlsx:
  Total rows: 250,931
  Unique FAY Status values: ['FAY', 'Not FAY', 'All']
  Value counts:
FAY Status
All        99032
FAY        95743
Not FAY    56156


In [14]:
def load_test_data_at_level(filepath, level):
    """Load one file's data at a given level (School/District/State)."""
    df = pd.read_excel(filepath, sheet_name=level)
    df = normalize_columns(df)
    
    if 'Subgroup' in df.columns:
        df = df[df['Subgroup'].isin(SUBGROUPS_OF_INTEREST)]
    if 'Test Level' in df.columns:
        df = df[df['Test Level'].isin(TEST_LEVELS_OF_INTEREST)]
    
    # FAY filter — use 'All' for inclusive count of all tested students
    # (matches Evan's statewide analysis methodology)
    # 2021 file has no FAY column, so filter is skipped (data passes through naturally)
    if 'FAY Status' in df.columns:
        df = df[df['FAY Status'] == 'All']
    
    df['source_file'] = filepath.name
    df['detected_year'] = extract_year_from_file(filepath)
    return df

# Reload
print("Loading test data... (this takes 1–2 minutes)\n")

datasets = {'School': [], 'District': [], 'State': []}

for f in test_files:
    year = extract_year_from_file(f)
    if year is None:
        continue
    for level in ['School', 'District', 'State']:
        try:
            df = load_test_data_at_level(f, level)
            datasets[level].append(df)
            print(f"  {f.name[:50]:50s} | {level:8s} | {len(df):>7,} rows")
        except Exception as e:
            print(f"  ERROR: {f.name} {level}: {e}")

schools_test = pd.concat(datasets['School'], ignore_index=True, sort=False)
districts_test = pd.concat(datasets['District'], ignore_index=True, sort=False)
state_test = pd.concat(datasets['State'], ignore_index=True, sort=False)

print(f"\nCOMBINED:")
print(f"  Schools:   {len(schools_test):>8,} rows")
print(f"  Districts: {len(districts_test):>8,} rows")
print(f"  State:     {len(state_test):>8,} rows")

print(f"\nRows per year:")
for level_name, df_level in [('Schools', schools_test), ('Districts', districts_test), ('State', state_test)]:
    if 'FiscalYear' in df_level.columns:
        print(f"\n  {level_name}:")
        for year, count in df_level.groupby('FiscalYear').size().items():
            print(f"    {year}: {count:,}")

Loading test data... (this takes 1–2 minutes)

  2021_Assessment_updated_redacted (1) (1).xlsx      | School   |  22,381 rows
  2021_Assessment_updated_redacted (1) (1).xlsx      | District |   6,961 rows
  2021_Assessment_updated_redacted (1) (1).xlsx      | State    |      56 rows
  math_ela_assessments_2022_redacted.xlsx            | School   |  34,022 rows
  math_ela_assessments_2022_redacted.xlsx            | District |  10,342 rows
  math_ela_assessments_2022_redacted.xlsx            | State    |      80 rows
  math_ela_assessments_2023_publish[1].xlsx          | School   |  34,468 rows
  math_ela_assessments_2023_publish[1].xlsx          | District |  10,521 rows
  math_ela_assessments_2023_publish[1].xlsx          | State    |      80 rows
  math_ela_assessmentsfy24 (2).xlsx                  | School   |  38,371 rows
  math_ela_assessmentsfy24 (2).xlsx                  | District |  11,799 rows
  math_ela_assessmentsfy24 (2).xlsx                  | State    |      88 rows
  mat

In [15]:
def clean_test_df(df):
    """Apply same suppression handling we used for grad rates."""
    df = df.copy()
    
    # Mark suppressed rows: '*' in any numeric column means suppressed
    numeric_cols_raw = [
        'Number Tested', 'Percent Passing',
        'Percent Proficiency Level 1', 'Percent Proficiency Level 2',
        'Percent Proficiency Level 3', 'Percent Proficiency Level 4',
    ]
    
    def is_suppressed(val):
        return isinstance(val, str) and ('*' in val or val.strip() in ('', '-', '—'))
    
    df['suppressed'] = df['Percent Passing'].apply(is_suppressed)
    
    # Coerce numeric columns to actual numbers, NaN for suppressed
    for col in numeric_cols_raw:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Standardize column names to snake_case
    df = df.rename(columns={
        'FiscalYear': 'fiscal_year',
        'Fiscal Year': 'fiscal_year',
        'County': 'county',
        'DistrictName': 'lea_name',
        'District Name': 'lea_name',
        'District Entity ID': 'lea_id',
        'DistrictCTDS': 'lea_ctds',
        'District CTDS': 'lea_ctds',
        'SchoolName': 'school_name',
        'School Name': 'school_name',
        'School Entity ID': 'school_id',
        'SchoolCTDS': 'school_ctds',
        'School CTDS': 'school_ctds',
        'Subject': 'subject',
        'Subgroup': 'subgroup',
        'Test Level': 'test_level',
        'FAY Status': 'fay_status',
        'Number Tested': 'n_tested',
        'Percent Passing': 'pct_passing',
        'Percent Proficiency Level 1': 'pct_level1',
        'Percent Proficiency Level 2': 'pct_level2',
        'Percent Proficiency Level 3': 'pct_level3',
        'Percent Proficiency Level 4': 'pct_level4',
    })
    
    return df

schools_test = clean_test_df(schools_test)
districts_test = clean_test_df(districts_test)
state_test = clean_test_df(state_test)

print(f"Schools:   {len(schools_test):,} rows | suppressed: {schools_test['suppressed'].sum():,} ({schools_test['suppressed'].mean()*100:.1f}%)")
print(f"Districts: {len(districts_test):,} rows | suppressed: {districts_test['suppressed'].sum():,} ({districts_test['suppressed'].mean()*100:.1f}%)")
print(f"State:     {len(state_test):,} rows | suppressed: {state_test['suppressed'].sum():,} ({state_test['suppressed'].mean()*100:.1f}%)")

# Verify the statewide trend matches your analysis
print(f"\nStatewide passing — All Students, All Assessments:")
mask = (state_test['subgroup'] == 'All Students') & (state_test['test_level'] == 'All Assessments')
state_check = state_test[mask].sort_values(['subject', 'fiscal_year'])
print(state_check[['fiscal_year', 'subject', 'pct_passing']].to_string(index=False))

Schools:   233,224 rows | suppressed: 142,231 (61.0%)
Districts: 71,966 rows | suppressed: 35,361 (49.1%)
State:     560 rows | suppressed: 9 (1.6%)

Statewide passing — All Students, All Assessments:
 fiscal_year               subject  pct_passing
        2021 English Language Arts         38.0
        2021 English Language Arts         15.0
        2021 English Language Arts         46.0
        2021 English Language Arts         36.0
        2022 English Language Arts         11.0
        2022 English Language Arts         46.0
        2022 English Language Arts         38.0
        2022 English Language Arts         40.0
        2023 English Language Arts         40.0
        2023 English Language Arts         11.0
        2023 English Language Arts         46.0
        2023 English Language Arts         39.0
        2024 English Language Arts         40.0
        2024 English Language Arts         10.0
        2024 English Language Arts         46.0
        2024 English Language A

In [16]:
mask = (state_test['subgroup'] == 'All Students') & (state_test['test_level'] == 'All Assessments')
state_check = state_test[mask].sort_values(['subject', 'fiscal_year'])
print("STATEWIDE — All Students × All Assessments only:")
print(state_check[['fiscal_year', 'subject', 'pct_passing', 'n_tested']].to_string(index=False))

STATEWIDE — All Students × All Assessments only:
 fiscal_year               subject  pct_passing  n_tested
        2021 English Language Arts         38.0  505124.0
        2021 English Language Arts         15.0    5655.0
        2021 English Language Arts         46.0  108496.0
        2021 English Language Arts         36.0  396628.0
        2022 English Language Arts         11.0    6734.0
        2022 English Language Arts         46.0  115620.0
        2022 English Language Arts         38.0  450112.0
        2022 English Language Arts         40.0  565516.0
        2023 English Language Arts         40.0  566787.0
        2023 English Language Arts         11.0    7485.0
        2023 English Language Arts         46.0  120617.0
        2023 English Language Arts         39.0  446422.0
        2024 English Language Arts         40.0  560174.0
        2024 English Language Arts         10.0    8527.0
        2024 English Language Arts         46.0  120621.0
        2024 English La

In [17]:
# Investigate: what other columns differentiate these rows?
mask = (state_test['subgroup'] == 'All Students') & (state_test['test_level'] == 'All Assessments') & (state_test['subject'] == 'English Language Arts') & (state_test['fiscal_year'] == 2025)
sample = state_test[mask]
print(f"{len(sample)} rows for ELA 2025 All Students All Assessments:")
print(sample.to_string())

4 rows for ELA 2025 All Students All Assessments:
                   subject      subgroup  fiscal_year       test_level District  n_tested  pct_passing  pct_level1  pct_level2  pct_level3  pct_level4                        source_file  detected_year  School Type fay_status  suppressed
305  English Language Arts  All Students         2025  All Assessments      NaN  559239.0         40.0        41.0        19.0        29.0        11.0  math_ela_assessmentsfy25 (5).xlsx           2025          All        All       False
327  English Language Arts  All Students         2025  All Assessments      NaN   11382.0         12.0        70.0        18.0        11.0         1.0  math_ela_assessmentsfy25 (5).xlsx           2025  Alternative        All       False
349  English Language Arts  All Students         2025  All Assessments      NaN  122014.0         45.0        36.0        19.0        32.0        13.0  math_ela_assessmentsfy25 (5).xlsx           2025      Charter        All       False
37

In [18]:
# Re-clean with School Type filter applied — keep only "All" (combined across school types)
# We still keep School Type column for later analysis if needed
def clean_test_df_v2(df):
    df = df.copy()
    
    # Mark suppression
    def is_suppressed(val):
        return isinstance(val, str) and ('*' in val or val.strip() in ('', '-', '—'))
    df['suppressed'] = df['Percent Passing'].apply(is_suppressed) if 'Percent Passing' in df.columns else False
    
    # Coerce numeric columns
    for col in ['Number Tested', 'Percent Passing', 'Percent Proficiency Level 1', 
                'Percent Proficiency Level 2', 'Percent Proficiency Level 3', 'Percent Proficiency Level 4']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    df = df.rename(columns={
        'FiscalYear': 'fiscal_year', 'Fiscal Year': 'fiscal_year',
        'County': 'county',
        'DistrictName': 'lea_name', 'District Name': 'lea_name',
        'District Entity ID': 'lea_id',
        'DistrictCTDS': 'lea_ctds', 'District CTDS': 'lea_ctds',
        'SchoolName': 'school_name', 'School Name': 'school_name',
        'School Entity ID': 'school_id',
        'SchoolCTDS': 'school_ctds', 'School CTDS': 'school_ctds',
        'Subject': 'subject', 'Subgroup': 'subgroup',
        'Test Level': 'test_level', 'FAY Status': 'fay_status',
        'School Type': 'school_type',
        'Number Tested': 'n_tested', 'Percent Passing': 'pct_passing',
        'Percent Proficiency Level 1': 'pct_level1',
        'Percent Proficiency Level 2': 'pct_level2',
        'Percent Proficiency Level 3': 'pct_level3',
        'Percent Proficiency Level 4': 'pct_level4',
    })
    return df

schools_test = clean_test_df_v2(schools_test)
districts_test = clean_test_df_v2(districts_test)
state_test = clean_test_df_v2(state_test)

# Verify state numbers match Evan's writeup
print("Statewide ELA + Math + Science, All Students, All Assessments, School Type = 'All':")
mask = (
    (state_test['subgroup'] == 'All Students') 
    & (state_test['test_level'] == 'All Assessments') 
    & (state_test['school_type'] == 'All')
)
state_check = state_test[mask].sort_values(['subject', 'fiscal_year'])
print(state_check[['fiscal_year', 'subject', 'pct_passing', 'n_tested']].to_string(index=False))

Statewide ELA + Math + Science, All Students, All Assessments, School Type = 'All':
 fiscal_year               subject  pct_passing  n_tested
        2022 English Language Arts         40.0  565516.0
        2023 English Language Arts         40.0  566787.0
        2024 English Language Arts         40.0  560174.0
        2025 English Language Arts         40.0  559239.0
        2022           Mathematics         33.0  571787.0
        2023           Mathematics         34.0  573016.0
        2024           Mathematics         32.0  568730.0
        2025           Mathematics         33.0  567984.0
        2022               Science         24.0  243076.0
        2023               Science         27.0  243174.0
        2024               Science         28.0  244207.0
        2025               Science         28.0  245485.0


In [19]:
# Diagnose 1: does 2021 have school_type?
print("=== 2021 columns ===")
df_2021 = state_test[state_test['fiscal_year'] == 2021]
print(f"Rows: {len(df_2021)}")
print(f"School Type values:", df_2021['school_type'].unique() if 'school_type' in df_2021.columns else 'COLUMN MISSING')
print()

# Diagnose 2: show ALL state-level ELA 2024 rows with all combinations of fay/school_type
print("=== ELA 2024 statewide All Students — all combinations ===")
mask = (
    (state_test['subgroup'] == 'All Students')
    & (state_test['test_level'] == 'All Assessments')
    & (state_test['fiscal_year'] == 2024)
    & (state_test['subject'] == 'English Language Arts')
)
sample = state_test[mask][['fay_status', 'school_type', 'n_tested', 'pct_passing']]
print(sample.to_string(index=False))

=== 2021 columns ===
Rows: 56
School Type values: <StringArray>
[nan]
Length: 1, dtype: str

=== ELA 2024 statewide All Students — all combinations ===
fay_status school_type  n_tested  pct_passing
       All         All  560174.0         40.0
       All Alternative    8527.0         10.0
       All     Charter  120621.0         46.0
       All    District  439553.0         38.0


In [20]:
# Re-verify with school_type == 'District' (matches Evan's writeup)
print("Statewide ELA + Math + Science — All Students, All Assessments, School Type = 'District':")
mask = (
    (state_test['subgroup'] == 'All Students')
    & (state_test['test_level'] == 'All Assessments')
    & (state_test['school_type'] == 'District')
)
state_check = state_test[mask].sort_values(['subject', 'fiscal_year'])
print(state_check[['fiscal_year', 'subject', 'pct_passing', 'n_tested']].to_string(index=False))

Statewide ELA + Math + Science — All Students, All Assessments, School Type = 'District':
 fiscal_year               subject  pct_passing  n_tested
        2022 English Language Arts         38.0  450112.0
        2023 English Language Arts         39.0  446422.0
        2024 English Language Arts         38.0  439553.0
        2025 English Language Arts         39.0  437225.0
        2022           Mathematics         32.0  454903.0
        2023           Mathematics         33.0  451094.0
        2024           Mathematics         32.0  446264.0
        2025           Mathematics         32.0  443558.0
        2022               Science         23.0  197689.0
        2023               Science         27.0  195792.0
        2024               Science         27.0  194977.0
        2025               Science         27.0  194855.0


In [21]:
# What test levels are actually in our cleaned data?
print("All unique test levels in district-level test data:")
print(districts_test['test_level'].value_counts().to_string())

All unique test levels in district-level test data:
test_level
All Assessments    71966


In [22]:
# Broaden the filter — we'll keep most test levels and decide downstream which to use for which analysis
TEST_LEVELS_OF_INTEREST = None  # Set to None to skip filtering at this stage; keep everything

def load_test_data_at_level_v2(filepath, level):
    df = pd.read_excel(filepath, sheet_name=level)
    df = normalize_columns(df)
    
    if 'Subgroup' in df.columns:
        df = df[df['Subgroup'].isin(SUBGROUPS_OF_INTEREST)]
    # Skip Test Level filtering — keep everything for now
    if 'FAY Status' in df.columns:
        df = df[df['FAY Status'] == 'All']
    
    df['source_file'] = filepath.name
    df['detected_year'] = extract_year_from_file(filepath)
    return df

print("Reloading test data with broader test level inclusion...\n")

datasets = {'School': [], 'District': [], 'State': []}
for f in test_files:
    year = extract_year_from_file(f)
    if year is None:
        continue
    for lvl in ['School', 'District', 'State']:
        try:
            df = load_test_data_at_level_v2(f, lvl)
            datasets[lvl].append(df)
        except Exception as e:
            print(f"  ERROR: {f.name} {lvl}: {e}")

schools_test = pd.concat(datasets['School'], ignore_index=True, sort=False)
districts_test = pd.concat(datasets['District'], ignore_index=True, sort=False)
state_test = pd.concat(datasets['State'], ignore_index=True, sort=False)

# Re-clean
schools_test = clean_test_df_v2(schools_test)
districts_test = clean_test_df_v2(districts_test)
state_test = clean_test_df_v2(state_test)

print(f"Schools:   {len(schools_test):>8,} rows")
print(f"Districts: {len(districts_test):>8,} rows")
print(f"State:     {len(state_test):>8,} rows")

print("\nUnique test levels in district data:")
print(districts_test['test_level'].value_counts().to_string())

Reloading test data with broader test level inclusion...

Schools:   1,024,989 rows
Districts:  395,519 rows
State:        7,110 rows

Unique test levels in district data:
test_level
All Assessments             71966
All Alt Assessments         18793
ELA Grade 6                 16426
ELA Grade 4                 16389
ELA Grade 3                 16285
ELA Grade 5                 16282
ELA Grade 7                 15752
ELA Grade 8                 15563
Math Grade 6                14225
Math Grade 4                14189
Math Grade 3                14097
Math Grade 5                14078
Science Grade 5             14038
Math Grade 7                13648
Math Grade 8                13475
Science Grade 8             13436
Math Grade 11                9788
Science Grade 11             9740
ELA Grade 11                 9710
Alt ELA Grade 5              3410
Alt ELA Grade 6              3376
Alt ELA Grade 4              3356
Alt ELA Grade 3              3259
Alt ELA Grade 7              3193
A

In [23]:
# Normalize test level names — collapse Mathematics → Math
def normalize_test_level(tl):
    if pd.isna(tl):
        return tl
    return str(tl).replace('Mathematics ', 'Math ').replace('Alt Mathematics ', 'Alt Math ')

for df in [schools_test, districts_test, state_test]:
    df['test_level'] = df['test_level'].apply(normalize_test_level)

print("Test levels after normalization (district data):")
print(districts_test['test_level'].value_counts().to_string())

Test levels after normalization (district data):
test_level
All Assessments         71966
All Alt Assessments     18793
Math Grade 6            16469
ELA Grade 6             16426
Math Grade 4            16426
ELA Grade 4             16389
Math Grade 3            16355
Math Grade 5            16313
ELA Grade 3             16285
ELA Grade 5             16282
Math Grade 7            15806
ELA Grade 7             15752
Math Grade 8            15621
ELA Grade 8             15563
Science Grade 5         14038
Science Grade 8         13436
Math Grade 11            9788
Science Grade 11         9740
ELA Grade 11             9710
Alt Math Grade 5         3437
Alt ELA Grade 5          3410
Alt Math Grade 6         3382
Alt ELA Grade 6          3376
Alt ELA Grade 4          3356
Alt Math Grade 4         3351
Alt Math Grade 3         3262
Alt ELA Grade 3          3259
Alt Math Grade 7         3196
Alt ELA Grade 7          3193
Alt Math Grade 8         3176
Alt ELA Grade 8          3169
Alt Scienc

In [25]:
print("Statewide Grade 11 — All Students, School Type = 'All':")
mask = (
    (state_test['subgroup'] == 'All Students')
    & (state_test['test_level'].isin(['ELA Grade 11', 'Math Grade 11']))
    & (state_test['school_type'] == 'All')
)
state_check = state_test[mask].sort_values(['test_level', 'fiscal_year'])
print(state_check[['fiscal_year', 'test_level', 'pct_passing', 'n_tested']].to_string(index=False))

print("\n2021 (school_type is NaN):")
mask_2021 = (
    (state_test['subgroup'] == 'All Students')
    & (state_test['test_level'].isin(['ELA Grade 11', 'Math Grade 11']))
    & (state_test['fiscal_year'] == 2021)
)
print(state_test[mask_2021][['test_level', 'pct_passing', 'n_tested', 'school_type']].to_string(index=False))

Statewide Grade 11 — All Students, School Type = 'All':
 fiscal_year    test_level  pct_passing  n_tested
        2022  ELA Grade 11         38.0   74429.0
        2023  ELA Grade 11         40.0   77640.0
        2024  ELA Grade 11         38.0   78323.0
        2025  ELA Grade 11         40.0   78765.0
        2022 Math Grade 11         32.0   76333.0
        2023 Math Grade 11         32.0   79801.0
        2024 Math Grade 11         30.0   82567.0
        2025 Math Grade 11         30.0   83811.0

2021 (school_type is NaN):
Empty DataFrame
Columns: [test_level, pct_passing, n_tested, school_type]
Index: []


In [26]:
# What test levels exist in the 2021 data specifically?
print("Test levels in 2021 state data:")
print(state_test[state_test['fiscal_year'] == 2021]['test_level'].value_counts().to_string())

Test levels in 2021 state data:
test_level
All Assessments        56
All Alt Assessments    52
ELA Grade 10           28
ELA Grade 3            28
ELA Grade 4            28
ELA Grade 5            28
ELA Grade 6            28
ELA Grade 7            28
ELA Grade 8            28
Math Grade 10          28
Math Grade 3           28
Math Grade 4           28
Math Grade 5           28
Math Grade 6           28
Math Grade 7           28
Math Grade 8           28
Alt ELA Grade 10       25
Alt ELA Grade 7        25
Alt Math Grade 10      25
Alt Math Grade 7       25
Alt ELA Grade 3        23
Alt ELA Grade 5        23
Alt ELA Grade 6        23
Alt Math Grade 3       23
Alt Math Grade 5       23
Alt Math Grade 6       23
Alt ELA Grade 4        22
Alt Math Grade 4       22
Alt ELA Grade 8        19
Alt Math Grade 8       19
Math Grade 1            4


In [28]:
# Final summary (skip school_type, which only exists at state level)
print(f"District-level coverage:")
print(f"  Years:       {sorted(districts_test['fiscal_year'].dropna().unique().tolist())}")
print(f"  Subjects:    {sorted(districts_test['subject'].dropna().unique().tolist())}")
print(f"  Test levels: {len(districts_test['test_level'].unique())}")
print(f"  Subgroups:   {sorted(districts_test['subgroup'].dropna().unique().tolist())}")
print(f"  Districts:   {districts_test['lea_id'].nunique()}")

print(f"\nState-level school_type values (only exists at state level):")
print(state_test['school_type'].value_counts(dropna=False).to_string())

District-level coverage:
  Years:       [2021, 2022, 2023, 2024, 2025]
  Subjects:    ['English Language Arts', 'Mathematics', 'Science']
  Test levels: 41
  Subgroups:   ['All Students', 'American Indian or Alaska Native', 'Asian', 'Black or African American', 'ELFEP14', 'Hispanic or Latino', 'Homeless', 'Income Eligibility 1 and 2', 'Limited English Proficient', 'Students with Disabilities', 'White']
  Districts:   669

State-level school_type values (only exists at state level):
school_type
District       1680
All            1680
Charter        1610
Alternative    1316
NaN             824


In [29]:
# How many districts are in each layer?
import pandas as pd
from pathlib import Path

trad = pd.read_csv('../data/reference/traditional_districts.csv')
districts_grad = pd.read_csv('../data/processed/districts.csv')
districts_test = pd.read_csv('../data/processed/test_scores_districts.csv')

print(f"NCES traditional districts (reference file):  {len(trad)}")
print(f"  ...with any grad rate data:                  {trad['lea_id'].isin(districts_grad['lea_id']).sum()}")
print(f"  ...with any test score data:                 {trad['lea_id'].isin(districts_test['lea_id']).sum()}")
print(f"\nAll districts in test score data:              {districts_test['lea_id'].nunique()}")
print(f"All districts in grad rate data:               {districts_grad['lea_id'].nunique()}")

NCES traditional districts (reference file):  214
  ...with any grad rate data:                  115
  ...with any test score data:                 208

All districts in test score data:              669
All districts in grad rate data:               419


/var/folders/ch/21gn10sn2yvfxds4zy7cz2l80000gn/T/ipykernel_80738/1134846835.py:7: DtypeWarning: Columns (0: fay_status) have mixed types. Specify dtype option on import or set low_memory=False.
  districts_test = pd.read_csv('../data/processed/test_scores_districts.csv')


In [30]:
# Confirm the analytical universe: traditional districts with BOTH grad rate AND Grade 11 test data
import pandas as pd

trad_ids = set(pd.read_csv('../data/reference/traditional_districts.csv')['lea_id'])
grad_ids = set(pd.read_csv('../data/processed/districts.csv')['lea_id'].unique())
test_ids = set(pd.read_csv('../data/processed/test_scores_districts.csv', low_memory=False)['lea_id'].unique())

# Filter test data to Grade 11 only — those are the districts with high schoolers being tested
districts_test = pd.read_csv('../data/processed/test_scores_districts.csv', low_memory=False)
g11_ids = set(districts_test[districts_test['test_level'].isin(['ELA Grade 11', 'Math Grade 11'])]['lea_id'].unique())

print(f"Universe layers:")
print(f"  Traditional districts (NCES):                {len(trad_ids)}")
print(f"  Traditional + has grad rate data:            {len(trad_ids & grad_ids)}")
print(f"  Traditional + has Grade 11 test data:        {len(trad_ids & g11_ids)}")
print(f"  Traditional + has BOTH grad AND G11 test:    {len(trad_ids & grad_ids & g11_ids)}")
print(f"\nThis last number is our analytical universe for the disparities analysis.")

Universe layers:
  Traditional districts (NCES):                214
  Traditional + has grad rate data:            115
  Traditional + has Grade 11 test data:        113
  Traditional + has BOTH grad AND G11 test:    113

This last number is our analytical universe for the disparities analysis.


In [31]:
# Which 2 districts have grad data but no Grade 11 test data?
in_grad_not_test = trad_ids & grad_ids - g11_ids
trad = pd.read_csv('../data/reference/traditional_districts.csv')
print(trad[trad['lea_id'].isin(in_grad_not_test)].to_string(index=False))

 lea_id                  district_name        county         city     nces_type
   4395         Cedar Unified District Navajo County KEAMS CANYON Regular Local
   4217 Graham County Special Services Graham County         PIMA Regular Local
